# Prediction of the Box-Counting Dimension of Metal Nanoparticles: Data Processing (Data-Driven Reduction)

This notebook contains the results from data processing in preparation for the machine learning predictions of the box-counting dimensions, $D_B$, computed for metal (specifically the mono-, bi-, and tri-metallic combinations of gold, palladium, and platinum) nanoparticles. The exercise is an attempt to explore the possibility of using a machine learning model to compute the $D_B$ values to save computational costs. The $D_B$ values are computed using [Sphractal](https://github.com/jon-ting/sphractal), a Python package for the estimation of the fractal dimension of the surfaces of atomistic objects via box-counting approaches.

## Outline

[Notebook Setups](#setup)

[Data Sets](#datasets)

[Monometallic Nanoparticles](#mnps)

<a id='setup'></a>
## Notebook Setups

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Import relevant libraries
from os import chdir, listdir
import pickle
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
# from sklearnex import patch_sklearn
# patch_sklearn()
import sklearn

from natsort import natsorted
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

chdir('/scratch/q27/jt5911/metal-nanoparticle-causal-path-probability/strategicInf')
from trainEvalModels import rmNullLowVarFeats, rmHighCorrFeats
from corrAnalysis import plotFeatsBCDcorrBar

# Print package versions for reproducibility
print('Versions of imported libraries:')
print(f"  matplotlib: {mpl.__version__}")
print(f"  numpy: {np.__version__}")
print(f"  pandas: {pd.__version__}")
print(f"  seaborn: {sns.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")

Versions of imported libraries:
  matplotlib: 3.8.2
  numpy: 1.24.4
  pandas: 2.1.4
  seaborn: 0.13.1
Versions of imported libraries:
  matplotlib: 3.8.2
  numpy: 1.24.4
  pandas: 2.1.4
  seaborn: 0.13.1
  scikit-learn: 1.4.0


Below are some general settings for plotting figures:

In [3]:
# Settings for figures
sns.set_theme(context='paper', style='ticks', palette='colorblind', font='sans-serif', font_scale=1, color_codes=True, rc=None)  # Options are: {paper, notebook, talk, poster}
figSize, DPI, fontSize, labelSize = (3.5, 2.5), None, 'medium', 'small'
legendSize, lineWidth, markerSize = 'x-small', 1, 3
# SMALL_SIZE, MEDIUM_SIZE, LARGE_SIZE, TITLE_SIZE = 8, 10, 12, 14
# plt.rc('font', size=LARGE_SIZE)  # controls default text sizes
# plt.rc('axes', titlesize=TITLE_SIZE)  # fontsize of the axes title
# plt.rc('axes', labelsize=LARGE_SIZE)  # fontsize of the x and y labels
# plt.rc('xtick', labelsize=LARGE_SIZE)  # fontsize of the tick labels
# plt.rc('ytick', labelsize=LARGE_SIZE)  # fontsize of the tick labels
# plt.rc('legend', fontsize=LARGE_SIZE)  # legend fontsize
# plt.rc('figure', titlesize=TITLE_SIZE)  # fontsize of the figure title
# PLOT_COLOURS = ['#212121', '#3F51B5', '#303F9F', '#FF5252', '#D32F2F']  # black, blue, deep blue, red, deep red from https://www.materialpalette.com
# mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=['#303F9F', '#FF5252', '#D32F2F'])

Some handy global variables:

In [4]:
VAR_THRESH = 0.0  # Value other than 0.0 is difficult to decide non-arbitrarily
VERBOSE = True
TRANS_PERC = False
# DATA_DIR = '/mnt/c/Users/ASUS/Documents/PhD/Workstation/PaperDrafts/metal-nanoparticle-causal-path-probability/data'
# FIG_DIR = '/mnt/c/Users/ASUS/Documents/PhD/Workstation/PaperDrafts/metal-nanoparticle-causal-path-probability/figs'
DATA_DIR = '/scratch/q27/jt5911/metal-nanoparticle-causal-path-probability/data'
FIG_DIR = '/scratch/q27/jt5911/metal-nanoparticle-causal-path-probability/figs'
RANDOM_STATE = 42
NUM_RUNS = 10

Some handy general functions:

In [5]:
def readPickle(picklePath):
    with open(picklePath, 'rb') as f: 
        df = pickle.load(f)
    df['rangeLenVX'] = df.apply(lambda f: f['maxLenVX'] - f['minLenVX'], axis=1)
    df['rangeLenEX'] = df.apply(lambda f: f['maxLenEX'] - f['minLenEX'], axis=1)
    return df

In [6]:
def loadCSV(csvFilePath, isMono=True, molLabels=True, ele='Au', transPerc=True):
    """
    The monometallic dataset has different naming for certain features, hence why 'isMono' needs to be specified.
    """
    featDF = pd.read_csv(csvFilePath, sep=',', header=0)

    if isMono:
        featDF = featDF.rename(columns={'N_total': 'N_atom_total', 'N_bulk': 'N_atom_bulk', 'N_surface': 'N_atom_surface', 
                                        'Avg_total': 'MM_TCN_avg', 'Avg_bulk': 'MM_BCN_avg', 'Avg_surf': 'MM_SCN_avg', 
                                        'Avg_bonds': 'BL_avg', 'Std_bonds': 'BL_std', 'Max_bonds': 'BL_max', 'Min_bonds': 'BL_min', 'N_bonds': 'BL_num',
                                        'angle_avg': 'BA1_avg', 'angle_std': 'BA1_std'}, 
                               inplace=False)
        eleSpecificFeats = [feat for feat in featDF.columns if 'CN_' in feat 
                                                            or 'BL_' in feat
                                                            or 'BA1_' in feat]
        # Rename and generate element-specific features to match bimetallic and trimetallic feature names
        eleSpecNewFeatNamesDict = {}
        for feat in eleSpecificFeats:
            eleSpecNewFeatName = f"MMM_{feat}" if 'BA1_' in feat else f"MM_{feat}"
            eleSpecNewFeatNamesDict[feat] = eleSpecNewFeatName
            eleSpecNewFeatName = f"{ele}{ele}{ele}_{feat}" if 'BA1_' in feat else f"{ele}{ele}_{feat}"
            featDF[eleSpecNewFeatName] = featDF[feat]
        featDF = featDF.rename(columns=eleSpecNewFeatNamesDict, inplace=False)
            
        # Rename q6q6 features to match bimetallic and trimetallic q6q6 feature names
        q6q6Feats = [feat for feat in featDF.columns if 'q6q6_T_' in feat or 'q6q6_B_' in feat or 'q6q6_S_' in feat]
        q6q6NewFeatNamesDict = {}
        for feat in q6q6Feats:
            q6q6NewFeatName = f"{feat[:6]}_{feat[6:]}"
            q6q6NewFeatNamesDict[feat] = q6q6NewFeatName
        featDF = featDF.rename(columns=q6q6NewFeatNamesDict, inplace=False)
    
    # Replace count features with percentages
    if transPerc:
        bulkAtomFeats = [feat for feat in featDF.columns if ('BCN' in feat and '_avg' not in feat)
                                                         or ('BGCN' in feat and '_avg' not in feat)
                                                         or 'q6q6_B_' in feat]
        surfAtomFeats = [feat for feat in featDF.columns if ('SCN' in feat and '_avg' not in feat)
                                                         or ('SGCN' in feat and '_avg' not in feat)
                                                         or ('SOCN' in feat and '_avg' not in feat)
                                                         or ('SOGCN' in feat and '_avg' not in feat)
                                                         or 'q6q6_S_' in feat 
                                                         or 'Curve' in feat
                                                         or 'S_100' in feat
                                                         or 'S_111' in feat
                                                         or 'S_110' in feat
                                                         or 'S_311' in feat]
        totAtomFeats = [feat for feat in featDF.columns if ('MM_TCN' in feat and '_avg' not in feat)
                                                        or ('TGCN' in feat and '_avg' not in feat)
                                                        or 'q6q6_T_' in feat
                                                        or 'FCC' in feat 
                                                        or 'HCP' in feat
                                                        or 'ICOS' in feat
                                                        or 'DECA' in feat]
        numBulkAtomFeat = featDF['N_atom_bulk']
        numSurfAtomFeat = featDF['N_atom_surface']
        numTotalAtomFeat = featDF['N_atom_total']
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for col in bulkAtomFeats:
                featDF[f"{col}"] = featDF[col] / numBulkAtomFeat * 100
            for col in surfAtomFeats:
                featDF[f"{col}"] = featDF[col] / numSurfAtomFeat * 100
            for col in totAtomFeats:
                featDF[f"{col}"] = featDF[col] / numTotalAtomFeat * 100
        featDF.drop(bulkAtomFeats, axis=1, inplace=True)
        featDF.drop(surfAtomFeats, axis=1, inplace=True)
        featDF.drop(totAtomFeats, axis=1, inplace=True)

    labelFeats = [feat for feat in featDF.columns if 'Surf_defects' in feat or 'Surf_micros' in feat or 'Surf_facets' in feat or '_E' in feat]
    labelDF = featDF[labelFeats]
    featDF.drop(['ID'] + labelFeats, axis=1, inplace=True)
    return featDF, labelDF

In [7]:
def loadFeats(pklFilePath):
    """
    Load feature dataframe saved in pickle file.
    """
    with open(pklFilePath, 'rb') as f:
        featsDF = pickle.load(f)
    return featsDF

<a id='datasets'></a>
## Datasets

The datasets used here are atomic coordinates of simulated metal nanoparticles generated for studies on the impact of polydispersivity on the properties of metal nanoparticle electrocatalyst ensembles. All data sets are hosted at [CSIRO Data Access Portal](https://data.csiro.au/collection/). The links are provided below:
- [gold](https://data.csiro.au/collection/csiro:40669)
- [palladium](https://data.csiro.au/collection/csiro:40618)
- [platinum](https://data.csiro.au/collection/csiro:36491)

<a id='mnps'></a>
## Monometallic Nanoparticles

Here we process the data for a machine learning model that aims to predict the $D_B$ values of a set of simulated monometallic nanoparticles (either gold, palladium, or platinum) to be trained on. We aim to ultimately study the feature importance profile to understand the influential features in the prediction of the $D_B$ values, and investigate the conditions that give rise to rougher nanoparticles.

### Data Loading

In [8]:
# Load features
MNPDF = loadFeats(f"{DATA_DIR}/MNPDF.pkl")
MNPFeatsDF, MNPtotEs, MNPformEs = MNPDF.iloc[:, :-2], MNPDF.iloc[:, -2], MNPDF.iloc[:, -1]
reducedMNPFeatsDF = loadFeats(f"{DATA_DIR}/reducedMNP_nanoparticle_data.pkl")
reducedMNPFeatsDF = reducedMNPFeatsDF.drop(['D_avg', 'Ele'], axis=1, inplace=False)

# # Load labels and extract useful label column(s)
feats = ['DBoxEX']  # 'R2EX', 'minLenEX', 'maxLenEX', 'rangeLenEX', 'lowCIEX', 'upCIEX'
AuLabelsDF = readPickle(f"{DATA_DIR}/allAuNPsEX.pickle")
AuLabelDF = AuLabelsDF[feats]
AuLabelDF = AuLabelDF.reset_index(inplace=False)
AuLabelDF = AuLabelDF.drop(['index'], axis=1, inplace=False)
PdLabelsDF = readPickle(f"{DATA_DIR}/allPdNPsEX.pickle")
PdLabelDF = PdLabelsDF[feats]
PdLabelDF = PdLabelDF.reset_index(inplace=False)
PdLabelDF = PdLabelDF.drop(['index'], axis=1, inplace=False)
PtLabelsDF = readPickle(f"{DATA_DIR}/allPtNPsEX.pickle")
PtLabelDF = PtLabelsDF[feats]
PtLabelDF = PtLabelDF.reset_index(inplace=False)
PtLabelDF = PtLabelDF.drop(['index'], axis=1, inplace=False)
MNPLabelDF = pd.concat([AuLabelDF, PdLabelDF, PtLabelDF], axis=0)
MNPLabelDF = MNPLabelDF.reset_index(inplace=False)
MNPLabelDF = MNPLabelDF.drop(['index'], axis=1, inplace=False)
MNPLabelDF = MNPLabelDF.iloc[MNPFeatsDF.index]

# Not using the original labels in this notebook, delete variables to free up memory
del AuLabelsDF, PdLabelsDF, PtLabelsDF

# for feat in MNPFeatsDF.columns:
#     print(feat)

In [9]:
### Data Processing

# Rearrange feature columns based on priorities to be kept (for the highly correlated feature removal step)
sortedFeats = ['T', 'time', 'tau',
               'MM_SCN_avg', 'q6q6_avg_surf', 
               'S_100', 'S_111', 'S_110', 'S_311', 
               'N_atom_surface', 'N_atom_bulk', 'Volume', 'N_atom_total', 
               'R_avg', 'R_diff', 'R_std', 'R_min', 'R_max', 'R_skew', 'R_kurt', 
               
               'MM_TCN_avg', 'MM_BCN_avg', 
               'MM_BL_avg', 'MM_BL_std', 'MM_BL_num', 'MM_BL_max', 'MM_BL_min', 
               'MMM_BA1_avg', 'MMM_BA1_std', 
               'q6q6_avg_total', 'q6q6_avg_bulk', 
                              
               'FCC', 'HCP', 'ICOS', 'DECA', 
               
               'Curve_1-10', 'Curve_11-20', 'Curve_21-30', 'Curve_31-40', 'Curve_41-50', 'Curve_51-60', 'Curve_61-70', 'Curve_71-80', 'Curve_81-90', 
               'Curve_91-100', 'Curve_101-110', 'Curve_111-120', 'Curve_121-130', 'Curve_131-140', 'Curve_141-150', 'Curve_151-160', 'Curve_161-170', 'Curve_171-180', 
               
               'MM_SCN_0', 'MM_SCN_1', 'MM_SCN_2', 'MM_SCN_3', 'MM_SCN_4', 'MM_SCN_5', 'MM_SCN_6', 'MM_SCN_7', 'MM_SCN_8', 'MM_SCN_9', 'MM_SCN_10', 
               'MM_SCN_11', 'MM_SCN_12', 'MM_SCN_13', 'MM_SCN_14', 'MM_SCN_15', 'MM_SCN_16', 'MM_SCN_17', 'MM_SCN_18', 'MM_SCN_19', 'MM_SCN_20', 
               'q6q6_S_0', 'q6q6_S_1', 'q6q6_S_2', 'q6q6_S_3', 'q6q6_S_4', 'q6q6_S_5', 'q6q6_S_6', 'q6q6_S_7', 'q6q6_S_8', 'q6q6_S_9', 'q6q6_S_10', 
               'q6q6_S_11', 'q6q6_S_12', 'q6q6_S_13', 'q6q6_S_14', 'q6q6_S_15', 'q6q6_S_16', 'q6q6_S_17', 'q6q6_S_18', 'q6q6_S_19', 'q6q6_S_20', 'q6q6_S_20+', 
               
               'MM_TCN_0', 'MM_TCN_1', 'MM_TCN_2', 'MM_TCN_3', 'MM_TCN_4', 'MM_TCN_5', 'MM_TCN_6', 'MM_TCN_7', 'MM_TCN_8', 'MM_TCN_9', 'MM_TCN_10', 
               'MM_TCN_11', 'MM_TCN_12', 'MM_TCN_13', 'MM_TCN_14', 'MM_TCN_15', 'MM_TCN_16', 'MM_TCN_17', 'MM_TCN_18', 'MM_TCN_19', 'MM_TCN_20', 
               'q6q6_T_0', 'q6q6_T_1', 'q6q6_T_2', 'q6q6_T_3', 'q6q6_T_4', 'q6q6_T_5', 'q6q6_T_6', 'q6q6_T_7', 'q6q6_T_8', 'q6q6_T_9', 'q6q6_T_10', 
               'q6q6_T_11', 'q6q6_T_12', 'q6q6_T_13', 'q6q6_T_14', 'q6q6_T_15', 'q6q6_T_16', 'q6q6_T_17', 'q6q6_T_18', 'q6q6_T_19', 'q6q6_T_20', 'q6q6_T_20+', 

               'MM_BCN_0', 'MM_BCN_1', 'MM_BCN_2', 'MM_BCN_3', 'MM_BCN_4', 'MM_BCN_5', 'MM_BCN_6', 'MM_BCN_7', 'MM_BCN_8', 'MM_BCN_9', 'MM_BCN_10', 
               'MM_BCN_11', 'MM_BCN_12', 'MM_BCN_13', 'MM_BCN_14', 'MM_BCN_15', 'MM_BCN_16', 'MM_BCN_17', 'MM_BCN_18', 'MM_BCN_19', 'MM_BCN_20', 
               'q6q6_B_0', 'q6q6_B_1', 'q6q6_B_2', 'q6q6_B_3', 'q6q6_B_4', 'q6q6_B_5', 'q6q6_B_6', 'q6q6_B_7', 'q6q6_B_8', 'q6q6_B_9', 'q6q6_B_10', 
               'q6q6_B_11', 'q6q6_B_12', 'q6q6_B_13', 'q6q6_B_14', 'q6q6_B_15', 'q6q6_B_16', 'q6q6_B_17', 'q6q6_B_18', 'q6q6_B_19', 'q6q6_B_20', 'q6q6_B_20+']
MNPFeatsDF = MNPFeatsDF[[feat for feat in sortedFeats if feat in MNPFeatsDF.columns]]

##### Feature Selection

In [10]:
reducedMNPFeatsDF = loadFeats(f"{DATA_DIR}/reducedMNP_nanoparticle_data.pkl")
# reducedMNPFeatsDF = reducedMNPFeatsDF.drop(['D_avg', 'Ele'], axis=1, inplace=False)
noArchMNPFeatsDF = MNPFeatsDF.drop(index=reducedMNPFeatsDF.index)
NUM_SAMPLES = np.linspace(250, noArchMNPFeatsDF.shape[0], NUM_RUNS, dtype=int)
NUM_SAMPLES

array([ 250, 1227, 2205, 3183, 4161, 5138, 6116, 7094, 8072, 9050])

In [13]:
CORR_THRESH = 0.8
numSamples = 250

for i in range(NUM_RUNS):
    # Feature selection
    featsDF = noArchMNPFeatsDF.sample(n=numSamples, random_state=RANDOM_STATE + i)
    # featsDF = pd.concat([featsDF, MNPFeatsDF.loc[reducedMNPFeatsDF.index]], axis=0)
    
    noLowVarDF = rmNullLowVarFeats(featsDF=featsDF, rmNull=False, varThresh=VAR_THRESH, verbose=VERBOSE)
    noLowVarHighCorrDF = rmHighCorrFeats(featsDF=noLowVarDF, corrThresh=CORR_THRESH, verbose=VERBOSE)
    del noLowVarDF
    
    # Merge labels with features
    MNPDF = pd.concat([noLowVarHighCorrDF, MNPLabelDF.loc[noLowVarHighCorrDF.index]], axis=1)
    # MNPDF = pd.concat([noLowVarHighCorrDF, MNPtotEs.loc[noLowVarHighCorrDF.index], MNPformEs.loc[noLowVarHighCorrDF.index]], axis=1)
    del noLowVarHighCorrDF

    # Data storage
    # with open(f"{DATA_DIR}/processedMNPdataNoC{int(CORR_THRESH * 100)}run{i+1}_250.pickle", 'wb') as f:
    with open(f"{DATA_DIR}/processedMNPdataNoC{int(CORR_THRESH * 100)}run{i+1}_250.pickle", 'wb') as f:
        pickle.dump(MNPDF, f)

Removing the features with variance below 0.00...
  Original number of features: 134
    MM_SCN_14:    0.000
    MM_SCN_15:    0.000
    MM_TCN_17:    0.000
    q6q6_T_15:    0.000
    MM_BCN_5:    0.000
    MM_BCN_17:    0.000
    q6q6_B_15:    0.000
  Total number of features left: 127

Removing the second feature from every pair of features with correlation above 0.80...
  Original number of features: 127
    q6q6_avg_surf MM_BL_std:    0.841
    q6q6_avg_surf MMM_BA1_std:    0.919
    q6q6_avg_surf q6q6_avg_total:    0.962
    q6q6_avg_surf q6q6_avg_bulk:    0.949
    S_111 Curve_1-10:    0.933
    S_111 MM_SCN_9:    0.931
    S_111 q6q6_S_9:    0.964
    S_111 MM_TCN_9:    0.925
    S_111 q6q6_T_9:    0.896
    S_110 MM_SCN_11:    0.878
    S_110 q6q6_S_11:    0.910
    S_311 q6q6_S_10:    0.861
    N_atom_surface N_atom_bulk:    0.956
    N_atom_surface N_atom_total:    0.971
    N_atom_surface R_avg:    0.977
    N_atom_surface R_max:    0.934
    N_atom_surface MM_BL_num:    0.

In [26]:
CORR_THRESH = 0.8

for (i, numSamples) in enumerate(NUM_SAMPLES):
    # Feature selection
    featsDF = noArchMNPFeatsDF.sample(n=numSamples, random_state=RANDOM_STATE + i)
    featsDF = pd.concat([featsDF, MNPFeatsDF.loc[reducedMNPFeatsDF.index]], axis=0)
    
    noLowVarDF = rmNullLowVarFeats(featsDF=featsDF, rmNull=False, varThresh=VAR_THRESH, verbose=VERBOSE)
    noLowVarHighCorrDF = rmHighCorrFeats(featsDF=noLowVarDF, corrThresh=CORR_THRESH, verbose=VERBOSE)
    del noLowVarDF
    
    # Merge labels with features
    # MNPDF = pd.concat([noLowVarHighCorrDF, MNPLabelDF.loc[noLowVarHighCorrDF.index]], axis=1)
    MNPDF = pd.concat([MNPFeatsNoLowVarHighCorrDF, MNPtotEs, MNPformEs], axis=1)
    del noLowVarHighCorrDF

    # Data storage
    # with open(f"{DATA_DIR}/processedMNPEdataNoC{int(CORR_THRESH * 100)}.pickle", 'wb') as f:
    with open(f"{DATA_DIR}/processedMNPdataNoC{int(CORR_THRESH * 100)}run{i+1}noArch.pickle", 'wb') as f:
        pickle.dump(MNPDF, f)

Removing the features with variance below 0.00...
  Original number of features: 134
  Total number of features left: 134

Removing the second feature from every pair of features with correlation above 0.80...
  Original number of features: 134
    q6q6_avg_surf MM_BL_std:    0.842
    q6q6_avg_surf MMM_BA1_std:    0.910
    q6q6_avg_surf q6q6_avg_total:    0.954
    q6q6_avg_surf q6q6_avg_bulk:    0.941
    S_100 q6q6_S_8:    0.816
    S_111 Curve_1-10:    0.905
    S_111 MM_SCN_9:    0.913
    S_111 q6q6_S_9:    0.952
    S_111 MM_TCN_9:    0.904
    S_111 q6q6_T_9:    0.888
    S_110 MM_SCN_11:    0.882
    S_110 q6q6_S_11:    0.895
    S_311 q6q6_S_10:    0.861
    N_atom_surface N_atom_bulk:    0.963
    N_atom_surface N_atom_total:    0.976
    N_atom_surface R_avg:    0.979
    N_atom_surface R_min:    0.832
    N_atom_surface R_max:    0.933
    N_atom_surface MM_BL_num:    0.972
    N_atom_surface Curve_21-30:    0.853
    N_atom_surface MM_SCN_10:    0.807
    N_atom_surface 

NameError: name 'MNPFeatsNoLowVarHighCorrDF' is not defined

In [25]:
CORR_THRESH = 0.8

for (i, numSamples) in enumerate(NUM_SAMPLES + 250):
    # Feature selection
    featsDF = MNPFeatsDF.sample(n=numSamples, random_state=RANDOM_STATE + i)
    noLowVarDF = rmNullLowVarFeats(featsDF=featsDF, rmNull=False, varThresh=VAR_THRESH, verbose=VERBOSE)
    noLowVarHighCorrDF = rmHighCorrFeats(featsDF=noLowVarDF, corrThresh=CORR_THRESH, verbose=VERBOSE)
    del noLowVarDF
    
    # Merge labels with features
    MNPDF = pd.concat([noLowVarHighCorrDF, MNPLabelDF.loc[noLowVarHighCorrDF.index]], axis=1)
    # MNPDF = pd.concat([MNPFeatsNoLowVarHighCorrDF, MNPtotEs, MNPformEs], axis=1)
    del noLowVarHighCorrDF

    # Data storage
    # with open(f"{DATA_DIR}/processedMNPEdataNoC{int(CORR_THRESH * 100)}.pickle", 'wb') as f:
    with open(f"{DATA_DIR}/processedMNPdataNoC{int(CORR_THRESH * 100)}run{i+1}.pickle", 'wb') as f:
        pickle.dump(MNPDF, f)

Removing the features with variance below 0.00...
  Original number of features: 134
    MM_SCN_15:    0.000
    MM_TCN_17:    0.000
    q6q6_T_15:    0.000
    MM_BCN_6:    0.000
    MM_BCN_17:    0.000
    q6q6_B_15:    0.000
  Total number of features left: 128

Removing the second feature from every pair of features with correlation above 0.80...
  Original number of features: 128
    q6q6_avg_surf MM_BL_std:    0.824
    q6q6_avg_surf MMM_BA1_std:    0.918
    q6q6_avg_surf q6q6_avg_total:    0.959
    q6q6_avg_surf q6q6_avg_bulk:    0.940
    S_100 q6q6_S_8:    0.814
    S_111 Curve_1-10:    0.876
    S_111 MM_SCN_9:    0.892
    S_111 q6q6_S_9:    0.949
    S_111 MM_TCN_9:    0.881
    S_111 q6q6_T_9:    0.856
    S_110 MM_SCN_11:    0.909
    S_110 q6q6_S_11:    0.932
    S_311 q6q6_S_10:    0.850
    N_atom_surface N_atom_bulk:    0.963
    N_atom_surface N_atom_total:    0.975
    N_atom_surface R_avg:    0.979
    N_atom_surface R_min:    0.844
    N_atom_surface R_max:    0